# 02 — Descriptive statistics

Characterises the two labelled cohorts before any modelling: per-variable means
and standard deviations split by insulin-resistance status, the tests behind
those comparisons, the distribution of each variable, the correlation structure,
and the distribution of the HOMA-IR index itself.

Produces thesis Tables `stats1`/`stats2`/`stats3`, Fig. `box-stats`,
Fig. `correlation` and Fig. `homa`.

Taiwan Biobank appears here only in the correlation panel. Its statistics table
and its boxplot need an `IR` label, which for that cohort is a *model
prediction* rather than a measurement, so both belong to stage 07.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import polars as pl

from src.data.io import output_path, processed_path
from src.logging_utils import configure_logging
from src.viz.figures import (
    plot_correlation_matrix,
    plot_feature_boxplots,
    plot_homair_distributions,
)
from src.viz.tables import descriptive_statistics

configure_logging(ROOT / "logs")

nhanes = pl.read_parquet(processed_path("NHANES_data.parquet"))
knhanes = pl.read_parquet(processed_path("KNHANES_data.parquet"))
twb = pl.read_parquet(processed_path("TWB_clinical_data.parquet"))
combined = pl.concat([nhanes, knhanes])

print(f"NHANES {nhanes.shape}, KNHANES {knhanes.shape}, combined {combined.shape}, TWB {twb.shape}")

NHANES (11660, 21), KNHANES (15138, 21), combined (26798, 21), TWB (92734, 18)


## Cohort characteristics

Each variable is reported as `mean±SD` for the whole cohort and for each
insulin-resistance group, followed by four p-values: a Kolmogorov–Smirnov test
of each group against a normal distribution, Welch's t-test, and the
Mann–Whitney U test.

One correction relative to the thesis code: the legacy IR− normality test
reported the p-value of a test run with the IR+ group's mean and standard
deviation whenever the result was not significant. The test is now run against
the IR− group's own moments throughout. As it turns out this changes nothing in
practice — every KS p-value in all three cohorts is below 0.01, so the faulty
branch was never reached — but the comparison is recorded in the gate below.

In [2]:
SHEETS = {"NHANES": nhanes, "KNHANES": knhanes, "COMBINE": combined}

tables = {name: descriptive_statistics(frame) for name, frame in SHEETS.items()}

with pd.ExcelWriter(output_path("stats.xlsx")) as writer:
    for name, table in tables.items():
        table.to_excel(writer, sheet_name=name, index=False)

tables["COMBINE"]

/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_distn_infrastructure.py:2071: RuntimeWarning: divide by zero encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_distn_infrastructure.py:2071: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
2026-09-15 23:00:08 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_distn_infrastructure.py:2071: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
2026-09-15 23:00:08 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_distn_infrastructure.py:2071: RuntimeWarning: divide by zero encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_distn_infrastructure.py:2071: RuntimeWarning: invalid value encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)
/opt/anaconda3/envs/bio/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
2026-09-15 23:00:08 [INFO] src.viz.tables: Descriptive statistics table: 20 rows


,column,All,IR-,IR+,ks_p_value-,ks_p_value+,t_test_p_value,u_test_p_value
0,N,"26,798","17,389","9,409",None,None,None,None
1,Male,"12,320(46.0%)","7,568(43.5%)","4,752(50.5%)",None,None,None,None
2,Female,"14,478(54.0%)","9,821(56.5%)","4,657(49.5%)",None,None,None,None
3,AGE,48.8±17.8,48.6±17.6,49.4±18.0,< 0.01,< 0.01,< 0.01,< 0.01
4,BMI,25.7±5.3,23.9±3.8,29.2±5.9,< 0.01,< 0.01,< 0.01,< 0.01
5,BODY_WAISTLINE,89.1±14.3,84.1±11.2,98.5±14.5,< 0.01,< 0.01,< 0.01,< 0.01
6,BUN,13.9±5.0,14.0±4.9,13.9±5.2,< 0.01,< 0.01,0.14433164293085082,< 0.01
7,CREATININE,0.8±0.3,0.8±0.3,0.8±0.3,< 0.01,< 0.01,< 0.01,< 0.01
8,FASTING_GLUCOSE,98.6±16.9,94.2±10.2,106.7±22.8,< 0.01,< 0.01,< 0.01,< 0.01
9,FASTING_INSULIN,10.5±9.8,6.2±2.3,18.5±13.0,< 0.01,< 0.01,< 0.01,< 0.01


## Distributions, correlations and the HOMA-IR index

The boxplot grid uses the `RACE` column as a cohort label on the x axis rather
than the numeric ancestry code. Five right-skewed variables are shown on a
natural-log scale.

One correction relative to the thesis figure: the cohort label reads `KNHANES`.
The published version reads `KNHAHES` in all fifteen panels, from a typo in the
literal that builds the label column.

In [3]:
boxplot_data = pl.concat(
    [
        nhanes.with_columns(RACE=pl.lit("NHANES")),
        knhanes.with_columns(RACE=pl.lit("KNHANES")),
    ]
).select(
    pl.all().exclude(
        ["Release_No", "DIABETES", "SEX", "MET_ID", "FASTING_INSULIN", "HOMA-IR"]
    )
)
boxplot_data = boxplot_data.select(sorted(boxplot_data.columns))

plot_feature_boxplots(
    boxplot_data,
    "Boxplots of variables by IR and Race",
    output_path("boxplot_of_variable_by_IR_and_Race.png"),
);


2026-09-15 23:00:10 [INFO] src.viz.figures: Wrote data/output/boxplot_of_variable_by_IR_and_Race.png


In [4]:
plot_correlation_matrix(
    {"NHANES": nhanes, "KNHANES": knhanes, "NHANES+KNHANES": combined, "TWB": twb},
    output_path("correlation_matrix.png"),
);


2026-09-15 23:00:11 [INFO] src.viz.figures: Wrote data/output/correlation_matrix.png


In [5]:
plot_homair_distributions(
    {"NHANES": nhanes, "KNHANES": knhanes, "NHANES+KNHANES": combined},
    output_path("HOMA-IR.png"),
);


2026-09-15 23:00:12 [INFO] src.viz.figures: Wrote data/output/HOMA-IR.png


## Gate G2

`stats.xlsx` is compared against the legacy workbook cell by cell, as text, so
that the `±` formatting and the row and column order are checked along with the
numbers. The `ks_p_value-` column is excluded from the pass criterion because a
decision deliberately changed how it is computed; its values are printed
separately so any movement is visible.

The figures have no numeric ground truth — the legacy project saved them only as
images — so they are compared pixel by pixel against `img/*.png`. Their inputs
are the stage 01 tables, which gate G1 already proved identical to the legacy
ones, so this comparison is a check on the plotting code rather than on the
data.

In [6]:
import numpy as np
from PIL import Image

from src.validate import compare_tables, reference_output, report

passed = True
for name, table in tables.items():
    reference = pd.read_excel(reference_output("stats.xlsx"), sheet_name=name)
    passed &= report(
        f"stats.xlsx[{name}]",
        compare_tables(table, reference, ignore_columns=["ks_p_value-"]),
    )
    moved = [
        (table["column"].iloc[row], reference["ks_p_value-"].iloc[row], table["ks_p_value-"].iloc[row])
        for row in range(len(table))
        if str(reference["ks_p_value-"].iloc[row]).strip() not in ("nan", str(table["ks_p_value-"].iloc[row]).strip())
    ]
    print(f"        ks_p_value- values changed by the correction: {len(moved)}")
    for column, before, after in moved:
        print(f"          {column}: {before!r} -> {after!r}")

print()
print("G2 (statistics tables):", "PASS" if passed else "FAIL")

2026-09-15 23:00:12 [INFO] src.validate: PASS stats.xlsx[NHANES]


2026-09-15 23:00:12 [INFO] src.validate: PASS stats.xlsx[KNHANES]


2026-09-15 23:00:12 [INFO] src.validate: PASS stats.xlsx[COMBINE]


PASS  stats.xlsx[NHANES]
        ks_p_value- values changed by the correction: 0
PASS  stats.xlsx[KNHANES]
        ks_p_value- values changed by the correction: 0
PASS  stats.xlsx[COMBINE]
        ks_p_value- values changed by the correction: 0

G2 (statistics tables): PASS


In [7]:
FIGURES = [
    "boxplot_of_variable_by_IR_and_Race.png",
    "correlation_matrix.png",
    "HOMA-IR.png",
]

for name in FIGURES:
    new = np.asarray(Image.open(output_path(name)).convert("RGB"), dtype=np.int16)
    legacy = np.asarray(Image.open(reference_output(name)).convert("RGB"), dtype=np.int16)
    if new.shape != legacy.shape:
        print(f"{name}: SIZE MISMATCH {new.shape} vs {legacy.shape}")
        continue
    differing = np.abs(new - legacy).sum(axis=2) > 0
    share = 100 * differing.sum() / differing.size
    print(f"{name}: {differing.sum():,} of {differing.size:,} pixels differ ({share:.2f}%)")

boxplot_of_variable_by_IR_and_Race.png: 690 of 2,250,000 pixels differ (0.03%)
correlation_matrix.png: 0 of 2,250,000 pixels differ (0.00%)


HOMA-IR.png: 0 of 2,250,000 pixels differ (0.00%)
